<a href="https://colab.research.google.com/github/Liping-LZ/BDAO_DSDO/blob/main/Week%201/06_BDAO_Block2_Pipeline_Student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# BDAO Block 2 — Student Notebook: Building a Data Pipeline
### UCI Online Retail Dataset · GCS · BigQuery

**How to use this notebook:**

| Symbol | Meaning |
|---|---|
| 📖 | Read and understand — no code to write |
| ▶️ | Just run this cell — no changes needed |
| ✏️ | Your turn — modify or add something |
| 💬 | Discussion — talk to the person next to you |
| 🤖 | Use AI — a suggested prompt is provided |

Follow your instructor's demo, then work through each stage yourself.
You are not expected to write code from scratch — modify, experiment, and understand.

**Pipeline you are building:**
```
SOURCE       DATA LAKE    TRANSFORM     WAREHOUSE      MART TABLES
UCI data  →  GCS      →  Python    →   BigQuery   →   3 analysis tables
```


---
## 📖 Before you start — set up Google Cloud

Google Cloud is a cloud computing platform. Today we will use three services:

| Service | What it is | Analogy |
|---|---|---|
| **GCP Project** | A container that holds all your cloud resources | A folder that holds everything |
| **Cloud Storage (GCS)** | A place to store files in the cloud | A hard drive in the cloud |
| **BigQuery** | A data warehouse for analysing large datasets with SQL | A smart spreadsheet that can handle millions of rows |

---
### Step A — Create a GCP Project

A **project** is a container for all your cloud resources. Think of it as a workspace.
Everything — your storage bucket, your BigQuery tables, your billing — lives inside a project.

1. Go to https://console.cloud.google.com
2. Click the project dropdown at the top → **New Project**
3. Give it a name (e.g. `bdao-dsdo`) → **Create**
4. Copy your **Project ID** — you will need it in Step 3 below
5. Enable billing for your project by linking a billing account (Please choose the “Billing account for Education”. Once you redeem the credits, this account should be automatically set up.)


> **Project ID** might be different from Project Name. It might looks like `bdao-dsdo-123456`.
> Find it on the project dashboard or in the URL.

---
### Step B — Create a Cloud Storage bucket

A **bucket** is a container for files in Cloud Storage — like a folder on a hard drive,
but accessible from anywhere and capable of storing terabytes of data.

This is where your raw data will land first — the **data lake**.

1. In the GCP console, search for **Cloud Storage** → open it
2. Click **Create bucket**
3. Give it a unique name (bucket names must be globally unique — try `bdao-yourname-2026`)
4. Region: **europe-west2 (London)** or **EU** multi-region
5. Leave all other settings as default → **Create**
6. Copy your **bucket name** — you will need it in Step 3 below

> **Note:** Bucket names cannot be changed after creation. Keep it simple.

---
### Step C — Create a BigQuery dataset

A **dataset** in BigQuery is a container for tables — like a database schema or a folder.
Your clean, analysis-ready tables will live inside this dataset.

1. In the GCP console, search for **BigQuery** → open it
2. In the left panel, click the three dots next to your project → **Create dataset**
3. Dataset ID: `retail_pipeline` (or any name you like — no spaces, use underscores)
4. Location: **EU** or **europe-west2**
5. Click **Create dataset**
6. Copy your **dataset name** — you will need it in Step 3 below

> **Tip:** Keep the region consistent — use the same region for your bucket and your dataset.
> Mixing regions causes errors when loading data from GCS to BigQuery.

---
### Step D — Enable required APIs

Before using a GCP service, you need to enable its API.
Search for each of these in the GCP console and click **Enable**:

- **Cloud Storage API**
- **BigQuery API**

If you get a permission error when running code, this is usually the cause.

---
Once you have completed Steps A–D, continue below.

---
## ▶️ Step 1 — Install and authenticate
Run these two cells first. The second one will open a popup — sign in with your Google account.

In [ ]:
# --quiet means: do not print all the installation details — just install silently
!pip install google-cloud-storage google-cloud-bigquery pandas-gbq ucimlrepo --quiet
print('Done.')

In [ ]:
from google.colab import auth
auth.authenticate_user()
print('Authenticated.')

---
## ✏️ Step 2 — Fill in your project details
Replace the three values below with what your instructor provides.

In [ ]:
# ── FILL IN YOUR DETAILS ──────────────────────────────────────
project_id   = 'your-project-id'     # ← replace with your GCP Project ID
bucket_name  = 'your-bucket-name'    # ← replace with your GCS bucket name
dataset_name = 'your-dataset-name'   # ← replace with your BigQuery dataset name
# ─────────────────────────────────────────────────────────────

# ── ALL IMPORTS ───────────────────────────────────────────────
# We import everything here so you always know what is available.
# If you copy code into your group project, copy these imports too.

import pandas as pd  # DataFrames — the main tool for working with tables
import numpy as np  # Numerical operations
import warnings
from datetime import datetime  # For timing the pipeline

from google.cloud import storage    # Cloud Storage — reading and writing files
from google.cloud import bigquery   # BigQuery — the data warehouse
from ucimlrepo import fetch_ucirepo # For fetching the UCI retail dataset today

warnings.filterwarnings('ignore')
pd.set_option('display.max_columns', 15)
pd.set_option('display.float_format', '{:.2f}'.format)

# ── CONNECT TO GCP ────────────────────────────────────────────
storage_client  = storage.Client(project=project_id)
bigquery_client = bigquery.Client(project=project_id)

print(f'Connected to GCP project: {project_id}')
print(f'Storage bucket:   gs://{bucket_name}')
print(f'BigQuery dataset: {project_id}.{dataset_name}')

---
## 📖 What we are building

Before writing any code, understand the flow:

**Stage 1 — Extract:** get the raw data from its source (UCI API today, GCS in your project)

**Stage 2 — Load raw:** save the untouched data to Cloud Storage (the data lake) before touching it

**Stage 3 — Validate:** understand what problems exist in the raw data before trying to fix them

**Stage 4 — Transform:** clean, fix and enrich the data — turn it into something useful

**Stage 5 — Load clean:** save the clean data to BigQuery (the warehouse)

**Stage 6 — Mart tables:** create focused summary tables for specific analytical purposes

**Stage 7 — Query:** ask business questions using SQL

**Stage 8 — Automate:** wrap everything into one function that can run on a schedule

---
## ▶️ Stage 1 — EXTRACT: Get the raw data

We fetch a real retail transaction dataset — 541,909 orders from a UK online gift store. Here is the link to the data: https://archive.ics.uci.edu/dataset/352/online+retail

This is our source data. In your group project, the source is the Yelp files in GCS.

In [ ]:
print('Fetching data...')
dataset = fetch_ucirepo(id=352)
df_raw = pd.concat([dataset.data.ids, dataset.data.features], axis=1)
print(f'Done — {df_raw.shape[0]:,} rows, {df_raw.shape[1]} columns')

In [ ]:
# Look at the first few rows to understand the structure
df_raw.head()

💬 **Quick check — before moving on:**
What does each column represent? Read the [data dictionary](https://archive.ics.uci.edu/dataset/352/online+retail).

---
## ▶️ Stage 2 — LOAD RAW to data lake (Cloud Storage)

**What is Cloud Storage?**
Google Cloud Storage (GCS) is like a hard drive in the cloud.
You store files in a **bucket** — think of it as a folder.
Files can be any format: CSV, JSON, images, videos.
Buckets can store terabytes of data cheaply.

**Why is it the data lake?**
A data lake stores raw, unprocessed data exactly as it arrived.
No cleaning, no transformation — just the original files.

The rule is: **always save raw data before touching it.**
If you make a mistake in cleaning, you can always come back to the original.
If you do not save it, you have to re-fetch from the source.

In [ ]:
# Save the raw data to Cloud Storage — untouched
json_data = df_raw.to_json(orient='records', lines=True)

bucket = storage_client.bucket(bucket_name)
blob   = bucket.blob('online_retail_raw.json')
blob.upload_from_string(json_data)

print(f'Saved to: gs://{bucket_name}/online_retail_raw.json') # dataset uri
print()
print(f'Check in Cloud Storage console:')
print(f'https://console.cloud.google.com/storage/browser/{bucket_name}')

---
## ▶️ Stage 3 — VALIDATE: Understand what we have

**Never trust incoming data.** Before cleaning anything, understand what problems exist.

Run each cell below and read the output carefully.
After all the checks, you will write down your decisions before any cleaning begins.

In [ ]:
# Check 1: Shape — how big is this dataset?
print('=== 1. SHAPE ===')
print(f'Rows:    {df_raw.shape[0]:,}')
print(f'Columns: {df_raw.shape[1]}')

In [ ]:
# Check 2: Data types — are the columns the right type?
# For example, dates should be 'datetime', numbers should be 'int' or 'float'
print('=== 2. DATA TYPES ===')
print(df_raw.dtypes)

💬 **Any issue with the data type?**

In [ ]:
# Check 3: Missing values — where are the gaps?
print('=== 3. MISSING VALUES ===')
missing     = df_raw.isnull().sum()
print(missing)

💬 **How to deal with missing values?**

In [ ]:
# Check 4: Duplicates — are any rows repeated exactly?
print('=== 4. DUPLICATES ===')
n_dupes = df_raw.duplicated().sum()
print(f'Fully duplicated rows: {n_dupes}')

💬 **How to deal with duplicates?**

In [ ]:
# Check 5: Descriptive statistics — what do the numbers look like?
# Look for: negative quantities, zero prices, extreme outliers
print('=== 5. DESCRIPTIVE STATISTICS ===')
df_raw.describe()

💬 **What do you found from this descriptive statistics?**

In [ ]:
# Check 6: Unique values — useful for spotting categories and outliers
print('=== 6. UNIQUE VALUES ===')
for col in df_raw.columns:
    n = df_raw[col].nunique()
    print(f'  {col:<20} {n:>7,} unique')

💬 **What is your observation here?**

In [ ]:
# Check 7: Spot checks — look for specific known issues in retail data
print('=== 7. SPOT CHECKS ===')
print()
cancels   = df_raw['InvoiceNo'].astype(str).str.startswith('C').sum()
neg_qty   = (pd.to_numeric(df_raw['Quantity'],  errors='coerce') < 0).sum()
zero_price= (pd.to_numeric(df_raw['UnitPrice'], errors='coerce') <= 0).sum()
print(f'Cancellations (InvoiceNo starts with C):  {cancels:,}')
print(f'Negative quantities:                      {neg_qty:,}')
print(f'Zero or negative unit prices:             {zero_price:,}')
print()
print('Countries in the data (top 10):')
print(df_raw['Country'].value_counts().head(10).to_string())

### ✏️ Validation complete — record your decisions

Before running any cleaning code, write down what you found and what you will do.
This is exactly what you need to explain in your group project presentation.

**Double-click to edit:**

| Issue found | What I will do | Why |
|---|---|---|
| InvoiceDate is a string not a date | | |
| CustomerID is stored as a float | | |
| Missing CustomerID (~25% of rows) | | |
| Duplicate rows | | |
| Cancellations (InvoiceNo starts with C) | | |
| Negative quantities | | |
| Zero/negative unit prices | | |

💬 **Compare with the person next to you** — did you make the same decisions?
Is there a case where two different decisions would both be valid?

---
## ▶️ Stage 4 — TRANSFORM: Clean and prepare

Now we act on the decisions above.
Each step is separate so you can see exactly what it does.

**As you run each step, ask yourself:**
- What does this line actually do?
- Why is it necessary?
- Would you do this differently for the Yelp data?

In [ ]:
# Start with a copy — NEVER modify the raw data directly
# If you make a mistake, you can always restart from df_raw
df = df_raw.copy()
rows_start = len(df)
print(f'Starting with {rows_start:,} rows.')
print('We will track how many rows each step removes.')

In [ ]:
# Step 4.1 — Fix data types
# What this does: tells Python what type each column actually is
# Why it matters: you cannot do date calculations on a string, or maths on an object

df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
# pd.to_datetime converts a string like "2011-12-01 08:26:00" into a proper date
# errors='coerce' means: if it cannot convert a value, put NaT (missing) instead of crashing

df['StockCode']   = df['StockCode'].astype(str).str.strip().str.upper()
df['Description'] = df['Description'].astype(str).str.strip().str.title()
df['Country']     = df['Country'].astype(str).str.strip().str.title()
# .str.strip() removes leading/trailing spaces
# .str.upper() makes everything UPPERCASE
# .str.title() makes First Letter Uppercase

df['CustomerID']  = df['CustomerID'].astype(str).str.replace(r'\.0$', '', regex=True)
# CustomerID was stored as 12345.0 (float) — we convert to '12345' (string)
# The regex r'\.0$' matches the '.0' at the end of the string and removes it

print('Step 4.1: data types fixed')
print(df.dtypes)

In [ ]:
# Step 4.2 — Remove duplicate rows
# What this does: removes rows that are exactly identical in every column
# Why: duplicate rows would double-count transactions in any aggregation

before = len(df)
df = df.drop_duplicates()
after  = len(df)
print(f'Step 4.2: {before - after:,} duplicate rows removed ({before:,} → {after:,})')

In [ ]:
# Step 4.3 — Remove rows with no date
# What this does: drops any row where InvoiceDate could not be converted to a date
# Why: a transaction without a date cannot be analysed by time — we cannot use it

before = len(df)
df     = df.dropna(subset=['InvoiceDate'])
after  = len(df)
print(f'Step 4.3: {before - after:,} rows without a valid date removed')

In [ ]:
# Step 4.4 — Flag cancellations (keep them, don't drop)
# What this does: adds a True/False column to mark cancelled orders
# Why keep them: cancellations are real business events
# Cancellation rate is itself an insight — we just need to know which rows they are

df['is_cancellation'] = df['InvoiceNo'].str.startswith('C')
# str.startswith('C') returns True if the InvoiceNo begins with the letter C

print(f'Step 4.4: {df["is_cancellation"].sum():,} cancellations flagged')
print(f'          {(~df["is_cancellation"]).sum():,} regular transactions')

In [ ]:
# Step 4.5 — Flag guest customers
# What this does: marks transactions where we do not know the customer
# Why: ~25% of transactions have no CustomerID — useful to know but not an error to remove

df['is_guest'] = df['CustomerID'] == 'nan'
# After the earlier type conversion, missing CustomerIDs became the string 'nan'

print(f'Step 4.5: {df["is_guest"].sum():,} guest transactions')
print(f'          {(~df["is_guest"]).sum():,} transactions with a known customer')

In [ ]:
# Step 4.6 — Flag returns (negative quantity)
# What this does: marks transactions where the quantity is negative
# Why: negative quantities are product returns — a customer sent items back
#      This is a real business event, not a data error
#      We keep them with a flag so we can analyse return rates later
#      Dropping them would hide important business information

df['is_return'] = (df['Quantity'] < 0)
# df['Quantity'] < 0 returns True if quantity is negative, False otherwise

print(f'Step 4.6: {df["is_return"].sum():,} return transactions flagged')
print(f'          {(~df["is_return"]).sum():,} regular sales transactions')

In [ ]:
# Step 4.7 — Flag zero or negative unit prices
# What this does: marks transactions where the price is zero or below
# Why: zero prices could be samples, gifts, or data entry errors
#      Negative prices are unusual and worth investigating separately
#      We flag rather than drop — they may be valid depending on your question
#      If your analysis is purely revenue-focused, you may choose to exclude them

df['is_zero_price'] = (df['UnitPrice'] == 0)
df['negative_price'] = (df['UnitPrice'] < 0)

# The flag_len variable is no longer useful as it was initially calculated incorrectly.
# We will calculate the counts directly in the print statements.

print(f'Step 4.7: {df["is_zero_price"].sum():,} zero price rows flagged')
print(f'          {df["negative_price"].sum():,} negative price rows flagged')
# Calculate rows with a positive price by excluding zero and negative prices
print(f'          {(df["UnitPrice"] > 0).sum():,} rows with a positive price')


In [ ]:
# Step 4.8 — Add derived columns
# What this does: creates new columns that do not exist in the raw data
# but are useful for analysis
# Why: revenue, month, day of week are all things we will want to analyse
# and it is easier to compute them once here than every time we query

df['Revenue']    = df['Quantity'] * df['UnitPrice']
# Revenue = how much money each transaction line generated

df['Year']       = df['InvoiceDate'].dt.year
df['Month']      = df['InvoiceDate'].dt.month
df['MonthLabel'] = df['InvoiceDate'].dt.strftime('%Y-%m')
# MonthLabel will look like '2011-12' — useful for grouping by month in order

df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
# day_name() returns 'Monday', 'Tuesday' etc.

df['Hour']       = df['InvoiceDate'].dt.hour
# Hour of the day (0-23) — useful for analysing when customers shop

print('Step 4.6: derived columns added')
print('New columns: Revenue, Year, Month, MonthLabel, DayOfWeek, Hour')

In [ ]:
# Step 4.9 — Choose which columns to keep
# What this does: keeps only the columns we actually need
# Why: dropping unused columns makes the table smaller and easier to work with
# In your group project, think carefully about which columns your question needs

columns_to_keep = [
    'InvoiceNo',       # order identifier
    'InvoiceDate',     # when the order was placed
    'MonthLabel',      # year-month label for grouping
    'Year',            # year number
    'Month',           # month number
    'DayOfWeek',       # day name
    'Hour',            # hour of day
    'StockCode',       # product code
    'Description',     # product name
    'Quantity',        # units ordered
    'UnitPrice',       # price per unit
    'Revenue',         # total revenue for this line
    'CustomerID',      # customer identifier
    'Country',         # customer country
    'is_cancellation', # True if this is a cancellation
    'is_guest',        # True if no CustomerID
    'is_return',       # True if the quantity is negative
    'is_zero_price',   # True if the price is zero
    'negative_price'   # True if the price is negative
]

df = df[columns_to_keep]
print(f'Step 4.9: kept {len(columns_to_keep)} columns')

In [ ]:
# Summary of all transform steps
print('=== TRANSFORM SUMMARY ===')
print(f'Rows before:  {rows_start:,}')
print(f'Rows after:   {len(df):,}')
print(f'Rows removed: {rows_start - len(df):,}')
print(f'Columns:      {df.shape[1]}')
print()
df.head(3)

### ✏️ Your turn — add your own transformation

Think about what other columns might be useful.
The examples below are commented out — uncomment one or write your own.

For each one, ask: **why would this be useful for analysis?**

In [ ]:
# YOUR CODE HERE:

print('Your transformation applied.')
df.head()

---
## ▶️ Stage 5 — LOAD clean table to BigQuery

**What is BigQuery?**
BigQuery is Google's cloud data warehouse — a database built for analysing large datasets.
Unlike the raw file in GCS, BigQuery tables are:
- **Structured** — columns have types, data is organised
- **Queryable** — you can run SQL to ask business questions
- **Persistent** — data survives when you close Colab
- **Shareable** — your whole group can query the same tables

**BigQuery concepts:**

| Term | What it is | Example |
|---|---|---|
| **Project** | Your GCP workspace | `bdao-block2-123456` |
| **Dataset** | A container for tables (like a folder) | `retail_pipeline` |
| **Table** | The actual data table | `retail_transactions` |
| **Full table path** | Project + Dataset + Table | `bdao-block2.retail_pipeline.retail_transactions` |

In SQL queries you always reference tables by their full path:
```sql
SELECT * FROM `project_id.dataset_name.table_name`
```

### Helper function: `load_to_bq()`

Before loading data to BigQuery, we define a helper function.
A helper function is a reusable block of code — instead of writing the same
loading code every time, we write it once and call it by name.

In [ ]:
def load_to_bq(df, table_name):
    """
    Save a DataFrame to BigQuery as a table.
    If the table already exists, it will be replaced.

    How to use:
        load_to_bq(my_dataframe, 'my_table_name')

    This will save the table to:
        project_id.dataset_name.my_table_name
    """
    df_bq = df.copy()

    # BigQuery cannot store pandas datetime objects directly
    # We convert them to strings first
    for col in df_bq.select_dtypes(include=['datetime64']).columns:
        df_bq[col] = df_bq[col].astype(str)

    # to_gbq() is the pandas function that writes to BigQuery
    df_bq.to_gbq(
        f'{dataset_name}.{table_name}',  # where to save: dataset.table
        project_id   = project_id,
        if_exists    = 'replace',        # replace the table if it already exists
        progress_bar = False
    )
    print(f'  Saved: {project_id}.{dataset_name}.{table_name} — {len(df_bq):,} rows')

print('load_to_bq() is ready to use.')

In [ ]:
# Create the BigQuery dataset (a container for tables) if it does not exist yet
dataset_ref = bigquery.DatasetReference(project_id, dataset_name)
dataset_obj = bigquery.Dataset(dataset_ref)
dataset_obj.location = 'EU'
try:
    bigquery_client.get_dataset(dataset_ref)
    print(f'Dataset already exists: {project_id}.{dataset_name}')
except Exception:
    bigquery_client.create_dataset(dataset_obj, timeout=30)
    print(f'Dataset created: {project_id}.{dataset_name}')

In [ ]:
# Load the clean transactions table to BigQuery
print('Loading clean transactions to BigQuery...')
load_to_bq(df, 'retail_transactions')
print()
print('View your table:')
print(f'https://console.cloud.google.com/bigquery?project={project_id}')

---
## ▶️ Stage 6 — MART TABLES: Create focused summary tables

One clean transactions table is good. But for analysis, **focused summary tables are better.**

A mart table is a pre-aggregated table designed for a specific analytical question:
- Want to analyse trends over time? → monthly revenue table
- Want to find best-selling products? → product summary table
- Want to understand customers? → customer summary table

We create all three from the same clean transactions table.
Each one is a different view of the same data, optimised for a different question.

**For your group project:** think about what mart table your analytical direction needs.
The Yelp equivalent might be: restaurants by city, reviews by business, users by engagement level.

In [ ]:
# Mart table 1: Monthly revenue summary
# Purpose: understand how the business performs over time
# One row per month — total orders, customers, revenue

mart_monthly = (
    df[~df['is_cancellation'] & ~df['is_guest']]   # exclude cancellations and guests
    .groupby('MonthLabel')                          # group all transactions by month
    .agg(
        orders          = ('InvoiceNo',  'nunique'), # count distinct orders
        customers       = ('CustomerID', 'nunique'), # count distinct customers
        total_revenue   = ('Revenue',    'sum'),     # sum all revenue
        avg_order_value = ('Revenue',    'mean')     # average revenue per transaction
    )
    .round(2)           # round to 2 decimal places
    .reset_index()      # turn the groupby index back into a column
)

print('Monthly revenue mart table:')
print(f'Shape: {mart_monthly.shape}')
mart_monthly.head()

In [ ]:
# Save the monthly mart table to BigQuery
load_to_bq(mart_monthly, 'mart_monthly_revenue')

In [ ]:
# Mart table 2: Product summary
# Purpose: understand which products sell best
# One row per product — total units, revenue, average price

mart_products = (
    df[~df['is_cancellation'] & (df['Quantity'] > 0)]  # real sales only
    .groupby(['StockCode', 'Description'])
    .agg(
        units_sold    = ('Quantity',  'sum'),
        avg_price     = ('UnitPrice', 'mean'),
        total_revenue = ('Revenue',   'sum'),
        times_ordered = ('InvoiceNo', 'nunique')
    )
    .round(2)
    .reset_index()
    .sort_values('total_revenue', ascending=False)  # highest revenue first
)

print('Product summary mart table:')
print(f'Shape: {mart_products.shape}')
mart_products.head()

In [ ]:
load_to_bq(mart_products, 'mart_product_summary')

In [ ]:
# Mart table 3: Customer summary
# Purpose: understand customer behaviour — who are our best customers?
# One row per customer — total spend, orders, first and last purchase

mart_customers = (
    df[~df['is_cancellation'] & ~df['is_guest']]   # known customers, no cancellations
    .groupby('CustomerID')
    .agg(
        country         = ('Country',     'first'),  # take the first country for this customer
        total_orders    = ('InvoiceNo',   'nunique'),
        total_items     = ('Quantity',    'sum'),
        total_spend     = ('Revenue',     'sum'),
        avg_order_value = ('Revenue',     'mean'),
        first_purchase  = ('InvoiceDate', 'min'),    # earliest order date
        last_purchase   = ('InvoiceDate', 'max')     # most recent order date
    )
    .round(2)
    .reset_index()
    .sort_values('total_spend', ascending=False)
)

print('Customer summary mart table:')
print(f'Shape: {mart_customers.shape}')
mart_customers.head()

In [ ]:
load_to_bq(mart_customers, 'mart_customer_summary')

For the Yelp dataset, what mart tables would your analytical direction need?

Think in this structure:

| Mart table name | One row per... | Columns it would have | Analytical question it answers |
|---|---|---|---|
| | | | |
| | | | |


💬 **Discuss with your group** — do you all need the same mart tables, or different ones?

---
## 🤖 Stage 7 — QUERY: Ask business questions

Now we query the mart tables using SQL.

**You do not need to know SQL syntax** — use AI to write the queries.
Your job is to know what question you want to answer, then read and interpret the result.

### How to use AI for SQL queries

Copy this prompt template and fill in the blanks:

```
Write a BigQuery SQL query that reads from the table
`PROJECT_ID.DATASET_NAME.TABLE_NAME` and shows
[what you want to see] grouped by [column name]
ordered by [column name] [ascending or descending]
limited to [number] rows.
```

Then paste the SQL into `run_query(""" YOUR SQL HERE """)` and run it.

### Helper function: `run_query()`

Before querying BigQuery, we define another helper function.
This one takes a SQL string, sends it to BigQuery, and returns the result
as a pandas DataFrame — a table you can work with in Python.

**Copy this function into your group project notebook too.**

In [ ]:
def run_query(sql):
    """
    Run a SQL query on BigQuery and return the results as a table.

    How to use:
        result = run_query(\"\"\"
            SELECT *
            FROM `project_id.dataset_name.table_name`
            LIMIT 10
        \"\"\")

    Returns a pandas DataFrame you can display, filter, or plot.
    """
    # bigquery_client.query() sends the SQL to BigQuery
    # .to_dataframe() converts the result into a pandas table
    return bigquery_client.query(sql).to_dataframe()

print('run_query() is ready to use.')

In [ ]:
# Pre-written query 1: What is the monthly revenue trend?
# Reading from the mart table — much simpler than querying the full transactions table

result = run_query(f"""
    SELECT MonthLabel, orders, total_revenue
    FROM `{project_id}.{dataset_name}.mart_monthly_revenue`
    ORDER BY MonthLabel
""")

print('Monthly revenue trend:')
result

In [ ]:
# Pre-written query 2: What are the top 10 products by revenue?

result = run_query(f"""
    SELECT Description, units_sold, total_revenue
    FROM `{project_id}.{dataset_name}.mart_product_summary`
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print('Top 10 products by revenue:')
result

In [ ]:
# Pre-written query 3: Which countries generate the most revenue?
# This one reads from the full transactions table with a GROUP BY

result = run_query(f"""
    SELECT
        Country,
        COUNT(DISTINCT CustomerID)  AS customers,
        ROUND(SUM(Revenue), 2)      AS total_revenue
    FROM `{project_id}.{dataset_name}.retail_transactions`
    WHERE is_cancellation = FALSE
    GROUP BY Country
    ORDER BY total_revenue DESC
    LIMIT 10
""")

print('Top 10 countries by revenue:')
result

### ✏️ 🤖 Your turn — write your own query using AI

Choose one of these questions — or ask your own:
- Which day of the week has the most orders?
- Which hour of the day has the highest revenue?
- Who are the top 5 customers by total spend?
- What is the overall cancellation rate?

**Use the AI prompt template above** to generate the SQL, then run it below.

In [ ]:
# ✏️ 🤖 Your question:

# Paste the AI-generated SQL below:
result = run_query(f"""
    -- PASTE YOUR SQL HERE
    SELECT *
    FROM `{project_id}.{dataset_name}.retail_transactions`
    LIMIT 5
""")

print('My result:')
result

---
## ▶️ Stage 8 — AUTOMATE: The full pipeline in one function

Everything we built step by step can be assembled into a **single reusable function**.

**Why does this matter?**
In a real business, data arrives continuously. You do not want to run
eight separate cells every time new data arrives. You want one function
that does everything automatically — every night, every hour, on demand.

This is also the structure your group project pipeline should follow.
You can copy this function and adapt it — replace the UCI data with Yelp data,
and replace the mart tables with ones that serve your question.

> Read through the function before running it.
> Can you recognise each stage from the steps above?

In [ ]:
def run_pipeline():
    """
    Full ELT pipeline — from source data to mart tables.

    Use this as a TEMPLATE for your group project pipeline.
    Replace:
      - Stage 1: fetch Yelp data from GCS instead of UCI
      - Stage 4: add transformations relevant to your question
      - Stage 6: create mart tables that serve your analytical direction
    """
    from datetime import datetime
    t0 = datetime.now()
    print(f'Pipeline started: {t0.strftime("%H:%M:%S")}')
    print('=' * 55)

    # ── Stage 1: EXTRACT ──────────────────────────────────────
    print('[1/6] Extracting raw data...')
    dataset = fetch_ucirepo(id=352)
    raw = pd.concat([dataset.data.ids, dataset.data.features], axis=1)
    print(f'      {len(raw):,} rows')

    # ── Stage 2: LOAD RAW TO LAKE ─────────────────────────────
    print('[2/6] Saving raw file to GCS (data lake)...')
    bucket = storage_client.bucket(bucket_name)
    blob   = bucket.blob('online_retail_raw.json')
    blob.upload_from_string(raw.to_json(orient='records', lines=True))
    print(f'gs://{bucket_name}/online_retail_raw.json')

    # ── Stage 3: VALIDATE ─────────────────────────────────────
    print('[3/6] Validating...')
    print(f'Missing: {raw.isnull().sum().sum():,} | Dupes: {raw.duplicated().sum():,}')

    # ── Stage 4: TRANSFORM ────────────────────────────────────
    print('[4/6] Transforming...')
    df = raw.copy()
    df['InvoiceDate'] = pd.to_datetime(df['InvoiceDate'], errors='coerce')
    df['InvoiceNo']   = df['InvoiceNo'].astype(str).str.strip()
    df['StockCode']   = df['StockCode'].astype(str).str.strip().str.upper()
    df['Description'] = df['Description'].astype(str).str.strip().str.title()
    df['Country']     = df['Country'].astype(str).str.strip().str.title()
    df['CustomerID']  = df['CustomerID'].astype(str).str.replace(r'\.0$','',regex=True)
    df = df.drop_duplicates().dropna(subset=['InvoiceDate'])
    df['is_cancellation'] = df['InvoiceNo'].str.startswith('C')
    df['is_guest']        = df['CustomerID'] == 'nan'
    df['is_return'] = (df['Quantity'] < 0)
    df['is_zero_price'] = (df['UnitPrice'] == 0)
    df['negative_price'] = (df['UnitPrice'] < 0)
    df['Revenue']    = df['Quantity'] * df['UnitPrice']
    df['MonthLabel'] = df['InvoiceDate'].dt.strftime('%Y-%m')
    df['Year']       = df['InvoiceDate'].dt.year
    df['Month']      = df['InvoiceDate'].dt.month
    df['DayOfWeek']  = df['InvoiceDate'].dt.day_name()
    df['Hour']       = df['InvoiceDate'].dt.hour
    df = df[['InvoiceNo','InvoiceDate','MonthLabel','Year','Month','DayOfWeek','Hour',
             'StockCode','Description','Quantity','UnitPrice','Revenue',
             'CustomerID','Country','is_cancellation','is_guest','is_return','is_zero_price','negative_price']]
    print(f'{len(raw):,} → {len(df):,} rows')

    # ── Stage 5: LOAD TO WAREHOUSE ────────────────────────────
    print('[5/6] Loading to BigQuery...')
    load_to_bq(df, 'retail_transactions')

    # ── Stage 6: MART TABLES ──────────────────────────────────
    print('[6/6] Producing mart tables...')

    clean = df[~df['is_cancellation'] & ~df['is_guest']]

    mart_m = (clean.groupby('MonthLabel')
              .agg(orders=('InvoiceNo','nunique'), customers=('CustomerID','nunique'),
                   total_revenue=('Revenue','sum'), avg_order_value=('Revenue','mean'))
              .round(2).reset_index())
    load_to_bq(mart_m, 'mart_monthly_revenue')

    mart_p = (df[~df['is_cancellation'] & (df['Quantity']>0)]
              .groupby(['StockCode','Description'])
              .agg(units_sold=('Quantity','sum'), avg_price=('UnitPrice','mean'),
                   total_revenue=('Revenue','sum'), times_ordered=('InvoiceNo','nunique'))
              .round(2).reset_index()
              .sort_values('total_revenue', ascending=False))
    load_to_bq(mart_p, 'mart_product_summary')

    mart_c = (clean.groupby('CustomerID')
              .agg(country=('Country','first'), total_orders=('InvoiceNo','nunique'),
                   total_spend=('Revenue','sum'), avg_order_value=('Revenue','mean'),
                   first_purchase=('InvoiceDate','min'), last_purchase=('InvoiceDate','max'))
              .round(2).reset_index()
              .sort_values('total_spend', ascending=False))
    load_to_bq(mart_c, 'mart_customer_summary')

    secs = (datetime.now() - t0).seconds
    print('=' * 55)
    print(f'Pipeline complete in {secs}s')
    print(f'Lake: gs://{bucket_name}/online_retail_raw.json')
    print(f'Tables: retail_transactions')
    print(f'mart_monthly_revenue')
    print(f'mart_product_summary')
    print(f'mart_customer_summary')


# Run the full pipeline in one call
run_pipeline()

---
## 📖 What you built today — and what it means for your group project

```
UCI dataset  →  GCS (lake)  →  Python (transform)  →  BigQuery (warehouse)
                                                             ↓
                                               retail_transactions
                                               mart_monthly_revenue
                                               mart_product_summary
                                               mart_customer_summary
```

**Your group project uses the same structure — just with Yelp data:**

| Today (UCI retail) | Your group project (Yelp) |
|---|---|
| `fetch_ucirepo(id=352)` | `pd.read_json('gs://yelp-data-ima/...')` |
| Save raw to GCS | Data already in GCS |
| Validate transactions | Validate restaurants + reviews |
| Transform + derive columns | Clean, join, add features for your question |
| `retail_transactions` | Your main joined table |
| `mart_monthly_revenue` | Your analysis-ready mart table |
| `run_pipeline()` | Your group's pipeline function |

**💬 Final discussion — with your group:**
1. Which Yelp datasets does your direction need?
2. What mart table would serve your analytical question best?
3. What derived columns will you need to create?
4. What data quality issues do you expect in the Yelp data?

